# Mini-Project 4: Text Classification Showdown

**Goal:** Compare classical ML vs transformers on the same text classification task.

**Dataset:** sklearn's 20 Newsgroups (4 categories)

**Categories:**
- `sci.space` — Space science
- `rec.sport.baseball` — Baseball
- `comp.graphics` — Computer graphics
- `talk.politics.guns` — Gun politics

**Estimated time:** ~2 hours

**What you'll learn:**
- TF-IDF text vectorization
- Classical text classifiers (Logistic Regression, Naive Bayes, Random Forest)
- Zero-shot classification with HuggingFace
- How to compare paradigms fairly

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Optional: HuggingFace (install with: pip install transformers)
# from transformers import pipeline as hf_pipeline

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load 20 Newsgroups — 4 distinct categories
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.guns']

train_data = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

print(f"Training samples: {len(train_data.data)}")
print(f"Test samples:     {len(test_data.data)}")
print(f"Categories:       {train_data.target_names}")
print(f"\n--- Sample document ---")
print(train_data.data[0][:300])

In [ ]:
# Results tracker — stores performance for each model
results = {}

def log_result(name, y_true, y_pred, train_time):
    """Log model results for later comparison."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    results[name] = {
        'accuracy': acc,
        'f1_macro': f1,
        'train_time': train_time,
    }
    print(f"\n{name}: accuracy={acc:.4f}, F1(macro)={f1:.4f}, time={train_time:.2f}s")

---
## Part 1: TF-IDF + Logistic Regression

Build a pipeline that vectorizes text with TF-IDF and classifies with Logistic Regression.

In [ ]:
# TODO: Build Pipeline(TfidfVectorizer(max_features=10000, ngram_range=(1,2)), LogisticRegression(max_iter=1000))
# TODO: Fit on training data, predict on test
# TODO: Print classification_report
# TODO: Log results with log_result()

# pipe_lr = Pipeline([
#     ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
#     ('clf', LogisticRegression(max_iter=1000)),
# ])
#
# start = time.time()
# pipe_lr.fit(train_data.data, train_data.target)
# train_time = time.time() - start
#
# preds_lr = pipe_lr.predict(test_data.data)
# print(classification_report(test_data.target, preds_lr, target_names=train_data.target_names))
# log_result("TF-IDF + LogReg", test_data.target, preds_lr, train_time)

---
## Part 2: TF-IDF + Naive Bayes

Multinomial Naive Bayes is a classic baseline for text classification. How does it compare?

In [ ]:
# TODO: from sklearn.naive_bayes import MultinomialNB
# TODO: Pipeline with TfidfVectorizer + MultinomialNB
# TODO: Evaluate and log

# pipe_nb = Pipeline([
#     ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
#     ('clf', MultinomialNB()),
# ])
#
# start = time.time()
# pipe_nb.fit(train_data.data, train_data.target)
# train_time = time.time() - start
#
# preds_nb = pipe_nb.predict(test_data.data)
# print(classification_report(test_data.target, preds_nb, target_names=train_data.target_names))
# log_result("TF-IDF + NaiveBayes", test_data.target, preds_nb, train_time)

---
## Part 3: TF-IDF + Random Forest

Random Forest with TF-IDF features. Does an ensemble approach help?

In [ ]:
# TODO: Pipeline with TfidfVectorizer + RandomForestClassifier
# TODO: Evaluate and log

# pipe_rf = Pipeline([
#     ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
#     ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
# ])
#
# start = time.time()
# pipe_rf.fit(train_data.data, train_data.target)
# train_time = time.time() - start
#
# preds_rf = pipe_rf.predict(test_data.data)
# print(classification_report(test_data.target, preds_rf, target_names=train_data.target_names))
# log_result("TF-IDF + RandomForest", test_data.target, preds_rf, train_time)

---
## Part 4: HuggingFace Zero-Shot Classification

No training needed! Use a pretrained zero-shot model to classify text into our categories.

**Note:** This will be much slower per sample than classical ML. Use a subset for evaluation.

In [ ]:
# TODO: Use pipeline("zero-shot-classification")
# candidate_labels = ["space science", "baseball sports", "computer graphics", "politics guns"]
# Classify test samples (use first 100 due to speed)
# TODO: Map predictions back to original labels
# TODO: Evaluate and log

# from transformers import pipeline as hf_pipeline
#
# zs_classifier = hf_pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
#
# candidate_labels = ["space science", "baseball sports", "computer graphics", "politics guns"]
# label_map = {
#     "space science": 3,       # index of sci.space in target_names
#     "baseball sports": 1,     # index of rec.sport.baseball
#     "computer graphics": 0,   # index of comp.graphics
#     "politics guns": 2,       # index of talk.politics.guns
# }
#
# n_samples = 100
# test_subset = test_data.data[:n_samples]
# true_labels = test_data.target[:n_samples]
#
# start = time.time()
# zs_preds = []
# for text in test_subset:
#     result = zs_classifier(text[:512], candidate_labels)  # truncate long texts
#     top_label = result['labels'][0]
#     zs_preds.append(label_map[top_label])
# train_time = time.time() - start
#
# zs_preds = np.array(zs_preds)
# print(classification_report(true_labels, zs_preds, target_names=train_data.target_names))
# log_result("Zero-Shot (BART)", true_labels, zs_preds, train_time)

---
## Part 5: Compare All Models

Visualize accuracy, F1 score, and training/inference time across all approaches.

In [ ]:
# Compare all models — bar chart
if results:
    df_results = pd.DataFrame(results).T
    df_results.index.name = 'Model'

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    df_results['accuracy'].plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Accuracy')
    axes[0].set_xlim(0, 1)

    df_results['f1_macro'].plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('F1 Score (Macro)')
    axes[1].set_xlim(0, 1)

    df_results['train_time'].plot(kind='barh', ax=axes[2], color='seagreen')
    axes[2].set_title('Time (seconds)')

    plt.tight_layout()
    plt.suptitle('Model Comparison', y=1.02, fontsize=14, fontweight='bold')
    plt.show()

    print("\nFull results table:")
    print(df_results.round(4))
else:
    print("No results logged yet! Complete Parts 1-4 first.")

---
## Part 6: Error Analysis

Show confusion matrix for the best classical model. Which categories are most confused?

In [ ]:
# TODO: Show confusion matrix for the best classical model
# - Which categories are most confused with each other?
# - Show a few misclassified examples

# Example (uncomment after completing Part 1):
# cm = confusion_matrix(test_data.target, preds_lr)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_data.target_names)
# fig, ax = plt.subplots(figsize=(8, 8))
# disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
# plt.title('Confusion Matrix — Best Classical Model')
# plt.tight_layout()
# plt.show()
#
# # Show misclassified examples
# misclassified = np.where(preds_lr != test_data.target)[0]
# print(f"\nTotal misclassified: {len(misclassified)} / {len(test_data.target)}")
# for idx in misclassified[:5]:
#     print(f"\n--- Misclassified #{idx} ---")
#     print(f"True: {train_data.target_names[test_data.target[idx]]}")
#     print(f"Pred: {train_data.target_names[preds_lr[idx]]}")
#     print(f"Text: {test_data.data[idx][:200]}...")

---
## Key Findings

**TODO — Fill in after completing all parts:**

1. **Best classical model:** (name and accuracy)
2. **Zero-shot vs trained:** How did the zero-shot model compare?
3. **Speed vs accuracy tradeoff:** Which model offers the best balance?
4. **Most confused categories:** Which two categories were hardest to distinguish?
5. **Biggest surprise:** What result did you not expect?

### Takeaways
- Classical ML with good features (TF-IDF + bigrams) is a strong baseline for text classification
- Zero-shot models require no training data but may sacrifice accuracy
- The right approach depends on your constraints: labeled data availability, latency requirements, accuracy needs